# Notebook 07 — Evaluation, Statistical Analysis & Additional Experiments
**Mental Health Assessment Using Machine Learning**

---

## Purpose

This notebook provides a comprehensive, statistically rigorous evaluation of the best model (Bayes XGB + SKB) and runs four additional experiments that strengthen the paper's methodology and findings.

---

## Best Model

| | |
|---|---|
| **Model** | XGBClassifier (Bayesian-optimised) — `models/Optimised/skb/xgb_bayes.pkl` |
| **Scaler** | `features/Tabular/skb/scaler.pkl` |
| **Test set** | `features/Tabular/skb/test.csv` (405 samples) |
| **Features (15)** | PSS2, PSS3, GAD1, GAD3–GAD7, PHQ2–PHQ8 |
| **Test F1_Macro** | 0.8667 · **Test Accuracy** 0.9086 |

---

## Experiments

**Cell 2 — ROC-AUC and Calibration Curves**  
One-vs-Rest ROC-AUC curves for all three classes (Stable, Challenged, Critical) with AUC scores, plus a macro-average curve. Calibration curves assess whether predicted probabilities are well-calibrated.

**Cell 3 — Statistical Significance Testing**  
Compares the best model (Bayes XGB + SKB) against the second-best (Bayes LGBM + FSCS, F1_Macro=0.8665 from NB04) using: (i) McNemar's test on test-set disagreements, and (ii) Wilcoxon signed-rank test on 5-fold CV F1_Macro scores. Both models use their Bayesian-optimised parameters from NB04; SMOTE is applied inside each CV fold.

**Cell 4 — Demographics Experiment**  
Model A (15 SKB scale items only) vs Model B (15 SKB scale items + 7 label-encoded demographics). Evaluated via Stratified 5-fold CV with SMOTE inside each fold. Tests whether demographic information adds predictive value beyond clinical scale items.

**Cell 5 — Ablation Study**  
Four progressive pipeline configurations evaluated via Stratified 5-fold CV:
1. All 26 scale items, no SMOTE, default XGB
2. SKB 15 features, no SMOTE, default XGB
3. SKB 15 features + SMOTE, default XGB (= best baseline)
4. SKB 15 features + SMOTE, Bayesian-optimised XGB

**Cell 6 — Multi-Output Classification**  
Instead of predicting a single `Mental Health Status`, predicts all three per-scale classes simultaneously (`Stress Level`, `Anxiety Level`, `Depression Level`) using `MultiOutputClassifier(XGB)`. Compared against the single-output approach using Hamming loss and exact match ratio.

---

## Primary Metric

**Macro F1** throughout — consistent with all previous notebooks.

---

## Cell Map

| Cell | Summary |
|------|---------|
| 1 | Imports · load best model, scaler, test set · verify F1_Macro |
| 2 | ROC-AUC curves + calibration curves · save figures |
| 3 | McNemar's test + Wilcoxon signed-rank · save `statistical_tests.csv` |
| 4 | Demographics experiment · save `demographics_experiment.csv` |
| 5 | Ablation study · save `ablation_study.csv` |
| 6 | Multi-output classification · save `multi_output_experiment.csv` |

## Cell 1 — Imports · Load Best Model · Verify

Loads the three best-model artefacts and verifies that F1_Macro and Accuracy match the values from NB04 (Bayesian optimisation). Also loads the original tabular dataset, which is needed for the demographics experiment, ablation study, and multi-output experiment — all of which require access to the full feature space and targets.

In [1]:
from pathlib import Path
import os, warnings, copy
warnings.filterwarnings('ignore')

_cwd = Path.cwd()
if _cwd.name == 'notebooks':
    os.chdir(_cwd.parent)
print(f'Working directory: {Path.cwd()}')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
import seaborn as sns
import joblib

from xgboost                   import XGBClassifier
from lightgbm                  import LGBMClassifier
from sklearn.multioutput       import MultiOutputClassifier
from sklearn.preprocessing     import LabelEncoder, StandardScaler, label_binarize
from sklearn.model_selection   import StratifiedKFold, cross_val_score
from sklearn.metrics           import (accuracy_score, f1_score, precision_score,
                                        recall_score, roc_curve, auc,
                                        classification_report)
from sklearn.calibration       import calibration_curve
from imblearn.over_sampling    import SMOTE
from statsmodels.stats.contingency_tables import mcnemar
from scipy.stats               import wilcoxon

plt.rcParams.update({'savefig.dpi': 300,
                     'axes.spines.top': False,
                     'axes.spines.right': False})
sns.set_palette('Set2')

CLASS_NAMES  = ['Stable', 'Challenged', 'Critical']
CLASS_COLORS = {'Stable': '#2ecc71', 'Challenged': '#f39c12', 'Critical': '#e74c3c'}
SEED         = 42
CV           = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

# ── Best model artefacts ───────────────────────────────────────────────────────
model  = joblib.load(os.path.join('models', 'Optimised', 'skb', 'xgb_bayes.pkl'))
scaler = joblib.load(os.path.join('features', 'Tabular', 'skb', 'scaler.pkl'))

test_df      = pd.read_csv(os.path.join('features', 'Tabular', 'skb', 'test.csv'))
FEAT_SKB     = [c for c in test_df.columns if c != 'label']
X_test       = test_df[FEAT_SKB].values
y_test       = test_df['label'].values

# ── Original tabular dataset (for experiments) ─────────────────────────────────
df = pd.read_csv(os.path.join('data', 'processed', 'mha_tabular_dataset.csv'))

# ── Verify ─────────────────────────────────────────────────────────────────────
y_pred   = model.predict(X_test)
y_proba  = model.predict_proba(X_test)
f1_macro = f1_score(y_test, y_pred, average='macro', zero_division=0)
acc      = accuracy_score(y_test, y_pred)
assert abs(f1_macro - 0.8667) < 0.001, f'F1_Macro mismatch: {f1_macro}'

print(f'Best model : {type(model).__name__} (Bayesian-optimised) + SKB')
print(f'Features   : {FEAT_SKB}')
print(f'F1_Macro   : {f1_macro:.4f}  (expected 0.8667)  ✓')
print(f'Accuracy   : {acc:.4f}  (expected 0.9086)')
print()
print(classification_report(y_test, y_pred, target_names=CLASS_NAMES, zero_division=0))

Working directory: d:\Programming\Projects\Mental Health Assessment
Best model : XGBClassifier (Bayesian-optimised) + SKB
Features   : ['PSS2', 'PSS3', 'GAD1', 'GAD3', 'GAD4', 'GAD5', 'GAD6', 'GAD7', 'PHQ2', 'PHQ3', 'PHQ4', 'PHQ5', 'PHQ6', 'PHQ7', 'PHQ8']
F1_Macro   : 0.8667  (expected 0.8667)  ✓
Accuracy   : 0.9086  (expected 0.9086)

              precision    recall  f1-score   support

      Stable       0.80      0.80      0.80        25
  Challenged       0.83      0.88      0.85       121
    Critical       0.96      0.93      0.95       259

    accuracy                           0.91       405
   macro avg       0.86      0.87      0.87       405
weighted avg       0.91      0.91      0.91       405



## Cell 2 — ROC-AUC Curves and Calibration Curves

**ROC-AUC:** Uses the One-vs-Rest (OvR) strategy — for each class, the class is treated as positive and all other classes as negative. One curve per class is plotted with its AUC score, plus a macro-average curve (unweighted mean of per-class AUCs). The Stable class (n=25) will have a noisier curve than Challenged or Critical due to the small sample size.

**Calibration curves:** A well-calibrated model produces predicted probabilities that match empirical frequencies — e.g., among samples predicted as Critical with 0.8 confidence, approximately 80% should actually be Critical. The diagonal reference line represents perfect calibration. `n_bins=8` is used to balance bin resolution against sample-count reliability, particularly for the Stable class.

In [2]:
FIG_DIR = os.path.join('figures', 'Evaluation')
os.makedirs(FIG_DIR, exist_ok=True)

y_test_bin = label_binarize(y_test, classes=[0, 1, 2])  # (405, 3)

# ── ROC-AUC curves ────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 6))
line_styles = ['-', '--', '-.']
auc_scores  = []

for i, (class_name, ls) in enumerate(zip(CLASS_NAMES, line_styles)):
    fpr, tpr, _ = roc_curve(y_test_bin[:, i], y_proba[:, i])
    auc_score   = auc(fpr, tpr)
    auc_scores.append(auc_score)
    ax.plot(fpr, tpr, linestyle=ls, linewidth=2,
            color=CLASS_COLORS[class_name],
            label=f'{class_name} (AUC = {auc_score:.4f})')

# Macro-average
all_fpr = np.unique(np.concatenate([
    roc_curve(y_test_bin[:, i], y_proba[:, i])[0] for i in range(3)
]))
mean_tpr = np.zeros_like(all_fpr)
for i in range(3):
    fpr_i, tpr_i, _ = roc_curve(y_test_bin[:, i], y_proba[:, i])
    mean_tpr += np.interp(all_fpr, fpr_i, tpr_i)
mean_tpr /= 3
macro_auc = auc(all_fpr, mean_tpr)
ax.plot(all_fpr, mean_tpr, 'k--', linewidth=2,
        label=f'Macro-Average (AUC = {macro_auc:.4f})')

ax.plot([0, 1], [0, 1], 'gray', linewidth=0.8, linestyle=':')
ax.set_title('ROC-AUC Curves — One-vs-Rest (Best Model: Bayes XGB + SKB)',
             fontsize=13, fontweight='bold')
ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
ax.legend(loc='lower right')
ax.set_xlim([0, 1]); ax.set_ylim([0, 1.02])
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'roc_auc_curves.png'), dpi=300, bbox_inches='tight')
plt.close()
print('✓ roc_auc_curves.png saved')
for name, score in zip(CLASS_NAMES, auc_scores):
    print(f'  {name:<12}: AUC = {score:.4f}')
print(f'  Macro-avg   : AUC = {macro_auc:.4f}')

# ── Calibration curves ────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 6))
ax.plot([0, 1], [0, 1], 'k:', linewidth=1, label='Perfect calibration')

for i, (class_name, ls) in enumerate(zip(CLASS_NAMES, line_styles)):
    if y_test_bin[:, i].sum() < 5:
        continue
    frac_pos, mean_pred = calibration_curve(
        y_test_bin[:, i], y_proba[:, i], n_bins=8, strategy='uniform'
    )
    ax.plot(mean_pred, frac_pos, marker='o', linestyle=ls, linewidth=2,
            color=CLASS_COLORS[class_name], label=class_name)

ax.set_title('Calibration Curves — Best Model: Bayes XGB + SKB',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Mean Predicted Probability')
ax.set_ylabel('Fraction of Positives')
ax.legend(loc='upper left')
ax.set_xlim([0, 1]); ax.set_ylim([0, 1])
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'calibration_curves.png'), dpi=300, bbox_inches='tight')
plt.close()
print('✓ calibration_curves.png saved')

✓ roc_auc_curves.png saved
  Stable      : AUC = 0.9833
  Challenged  : AUC = 0.9661
  Critical    : AUC = 0.9856
  Macro-avg   : AUC = 0.9792
✓ calibration_curves.png saved


## Cell 3 — Statistical Significance Testing

Tests whether the best model (Bayes XGB + SKB) is statistically significantly better than the second-best (Bayes LGBM + FSCS, F1_Macro=0.8665 from NB04 Bayesian optimisation) using two complementary tests:

**McNemar's test:** Operates on the 2×2 contingency table of test-set disagreements. Counts how often the best model is correct while the second-best is wrong (b) vs the reverse (c). The exact Binomial version is used since b+c may be small. Both models predict on the same 405 test samples (same stratified split with `random_state=42`).

**Wilcoxon signed-rank test:** Compares paired F1_Macro scores across 5 CV folds for both models on the original dataset. SMOTE is applied inside each fold to the training portion only. The Wilcoxon test is non-parametric and appropriate since we cannot assume normality with only 5 pairs.

In [3]:
# ── Load second-best model (Bayes LGBM + FSCS) ────────────────────────────────
model_2nd  = joblib.load(os.path.join('models', 'Optimised', 'fscs', 'lgbm_bayes.pkl'))
test_2nd   = pd.read_csv(os.path.join('features', 'Tabular', 'fscs', 'test.csv'))
FEAT_FSCS  = [c for c in test_2nd.columns if c != 'label']
X_test_2nd = test_2nd[FEAT_FSCS].values
y_test_2nd = test_2nd['label'].values

y_pred_best   = model.predict(X_test)
y_pred_second = model_2nd.predict(X_test_2nd)

f1_best   = f1_score(y_test,     y_pred_best,   average='macro', zero_division=0)
f1_second = f1_score(y_test_2nd, y_pred_second, average='macro', zero_division=0)
print(f'Best   (Bayes XGB+SKB)  : F1_Macro={f1_best:.4f}')
print(f'Second (Bayes LGBM+FSCS): F1_Macro={f1_second:.4f}')

# ── McNemar's test ─────────────────────────────────────────────────────────────
best_correct   = (y_pred_best   == y_test)
second_correct = (y_pred_second == y_test_2nd)

a = int(np.sum( best_correct &  second_correct))
b = int(np.sum( best_correct & ~second_correct))
c = int(np.sum(~best_correct &  second_correct))
d = int(np.sum(~best_correct & ~second_correct))
table = [[a, b], [c, d]]

print(f'\nMcNemar contingency table:')
print(f'  Both correct       (a): {a}')
print(f'  Best only correct  (b): {b}')
print(f'  Second only correct(c): {c}')
print(f'  Both wrong         (d): {d}')

mn_result = mcnemar(table, exact=True, correction=True)
print(f'\nMcNemar test (exact=True, correction=True):')
print(f'  Statistic : {mn_result.statistic:.4f}')
print(f'  p-value   : {mn_result.pvalue:.4f}')
print(f'  Significant (p<0.05): {mn_result.pvalue < 0.05}')

# ── Wilcoxon signed-rank test — 5-fold CV with Bayesian-optimised parameters ──
TARGET_MAP  = {'Stable': 0, 'Challenged': 1, 'Critical': 2}
SCALE_ITEMS = ([f'PSS{i}' for i in range(1, 11)] +
               [f'GAD{i}' for i in range(1,  8)] +
               [f'PHQ{i}' for i in range(1, 10)])
X_all = df[SCALE_ITEMS].values
y_all = df['Mental Health Status'].map(TARGET_MAP).values

SKB_IDX  = [SCALE_ITEMS.index(f) for f in FEAT_SKB]    # best-model features
FSCS_IDX = [SCALE_ITEMS.index(f) for f in FEAT_FSCS]   # second-best features

# Extract Bayesian-optimised parameters (exclude infrastructure keys)
_EXCL = {'n_jobs', 'random_state', 'verbosity', 'eval_metric',
          'early_stopping_rounds', 'callbacks', 'silent',
          'importance_type', 'class_weight'}
xgb_cv_params  = {k: v for k, v in model.get_params().items()
                  if k not in _EXCL}
lgbm_cv_params = {k: v for k, v in model_2nd.get_params().items()
                  if k not in _EXCL}

scores_best, scores_second = [], []
smote = SMOTE(random_state=SEED)

for fold, (tr_idx, val_idx) in enumerate(CV.split(X_all, y_all), 1):
    y_tr, y_val = y_all[tr_idx], y_all[val_idx]

    # Best: Bayes XGB + SKB features
    X_tr_b  = X_all[tr_idx][:, SKB_IDX]
    X_val_b = X_all[val_idx][:, SKB_IDX]
    X_tr_sm_b, y_tr_sm_b = smote.fit_resample(X_tr_b, y_tr)
    sc_b = StandardScaler()
    X_tr_sc_b  = sc_b.fit_transform(X_tr_sm_b)
    X_val_sc_b = sc_b.transform(X_val_b)
    clf_b = XGBClassifier(**xgb_cv_params, random_state=SEED, n_jobs=-1,
                           eval_metric='mlogloss', verbosity=0)
    clf_b.fit(X_tr_sc_b, y_tr_sm_b)
    score_b = f1_score(y_val, clf_b.predict(X_val_sc_b), average='macro', zero_division=0)
    scores_best.append(score_b)

    # Second: Bayes LGBM + FSCS features
    X_tr_f  = X_all[tr_idx][:, FSCS_IDX]
    X_val_f = X_all[val_idx][:, FSCS_IDX]
    X_tr_sm_f, y_tr_sm_f = smote.fit_resample(X_tr_f, y_tr)
    sc_f = StandardScaler()
    X_tr_sc_f  = sc_f.fit_transform(X_tr_sm_f)
    X_val_sc_f = sc_f.transform(X_val_f)
    clf_f = LGBMClassifier(**lgbm_cv_params, random_state=SEED, n_jobs=-1, verbosity=-1)
    clf_f.fit(X_tr_sc_f, y_tr_sm_f)
    score_f = f1_score(y_val, clf_f.predict(X_val_sc_f), average='macro', zero_division=0)
    scores_second.append(score_f)

    print(f'  Fold {fold}: SKB(XGB)={score_b:.4f}  FSCS(LGBM)={score_f:.4f}')

wil_stat, wil_p = wilcoxon(scores_best, scores_second, alternative='greater')
print(f'\nWilcoxon signed-rank test (alternative=greater):')
print(f'  SKB(XGB)   CV F1_Macro: {[round(s,4) for s in scores_best]}')
print(f'  FSCS(LGBM) CV F1_Macro: {[round(s,4) for s in scores_second]}')
print(f'  Statistic : {wil_stat:.4f}')
print(f'  p-value   : {wil_p:.4f}')
print(f'  Significant (p<0.05): {wil_p < 0.05}')

# ── Save statistical_tests.csv ────────────────────────────────────────────────
os.makedirs(os.path.join('summary', 'Evaluation'), exist_ok=True)
stat_df = pd.DataFrame([{
    'Best_Model'        : 'Bayes XGB+SKB',
    'Second_Model'      : 'Bayes LGBM+FSCS',
    'Best_F1_Macro'     : round(f1_best,   4),
    'Second_F1_Macro'   : round(f1_second, 4),
    'McNemars_Statistic': round(mn_result.statistic, 4),
    'McNemars_p'        : round(mn_result.pvalue, 4),
    'Wilcoxon_Statistic': round(wil_stat, 4),
    'Wilcoxon_p'        : round(wil_p, 4),
    'SKB_CV_Mean'       : round(np.mean(scores_best),   4),
    'FSCS_CV_Mean'      : round(np.mean(scores_second), 4),
}])
STAT_PATH = os.path.join('summary', 'Evaluation', 'statistical_tests.csv')
stat_df.to_csv(STAT_PATH, index=False)
print(f'\n✓ Saved: {STAT_PATH}')

Best   (Bayes XGB+SKB)  : F1_Macro=0.8667
Second (Bayes LGBM+FSCS): F1_Macro=0.8665

McNemar contingency table:
  Both correct       (a): 357
  Best only correct  (b): 11
  Second only correct(c): 12
  Both wrong         (d): 25

McNemar test (exact=True, correction=True):
  Statistic : 11.0000
  p-value   : 1.0000
  Significant (p<0.05): False
  Fold 1: SKB(XGB)=0.8386  FSCS(LGBM)=0.7991
  Fold 2: SKB(XGB)=0.8496  FSCS(LGBM)=0.8380
  Fold 3: SKB(XGB)=0.8802  FSCS(LGBM)=0.8389
  Fold 4: SKB(XGB)=0.8944  FSCS(LGBM)=0.8227
  Fold 5: SKB(XGB)=0.8351  FSCS(LGBM)=0.8409

Wilcoxon signed-rank test (alternative=greater):
  SKB(XGB)   CV F1_Macro: [0.8386, 0.8496, 0.8802, 0.8944, 0.8351]
  FSCS(LGBM) CV F1_Macro: [0.7991, 0.838, 0.8389, 0.8227, 0.8409]
  Statistic : 14.0000
  p-value   : 0.0625
  Significant (p<0.05): False

✓ Saved: summary\Evaluation\statistical_tests.csv


## Cell 4 — Demographics Experiment

Tests whether including demographic features (age group, gender, university, department, year of study, CGPA range, scholarship status) improves classification beyond the 15 SKB-selected scale items alone.

**Model A:** XGB on the 15 SKB scale features (current best pipeline)  
**Model B:** XGB on the same 15 scale features + 7 label-encoded demographic features (22 features total)

Both models use identical XGB parameters and are evaluated via Stratified 5-fold CV with SMOTE applied inside each fold to the training portion only. Results include per-class F1, macro F1, weighted F1, and accuracy.

In [4]:
DEMO_COLS = ['Age','Gender','University','Department','Year','CGPA','Scholarship']

# Label-encode demographics
df_demo = df.copy()
for col in DEMO_COLS:
    df_demo[col] = LabelEncoder().fit_transform(df_demo[col].astype(str))

X_scale = df[SCALE_ITEMS].values
X_demo  = df_demo[DEMO_COLS].values
y_all   = df['Mental Health Status'].map(TARGET_MAP).values

# SKB indices within the 26-item scale array
SKB_IDX = [SCALE_ITEMS.index(f) for f in FEAT_SKB]

def cv_evaluate(X, y, model_name, n_splits=5):
    """5-fold CV with SMOTE inside each fold. Returns results dict."""
    cv_local = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=SEED)
    sm = SMOTE(random_state=SEED)
    fold_f1m, fold_f1w, fold_acc = [], [], []
    per_class = {c: [] for c in CLASS_NAMES}

    for tr_idx, val_idx in cv_local.split(X, y):
        X_tr, X_val = X[tr_idx], X[val_idx]
        y_tr, y_val = y[tr_idx], y[val_idx]
        X_tr_sm, y_tr_sm = sm.fit_resample(X_tr, y_tr)
        sc = StandardScaler()
        X_tr_sc  = sc.fit_transform(X_tr_sm)
        X_val_sc = sc.transform(X_val)
        clf = XGBClassifier(n_estimators=100, random_state=SEED, n_jobs=-1,
                            eval_metric='mlogloss', verbosity=0)
        clf.fit(X_tr_sc, y_tr_sm)
        y_p = clf.predict(X_val_sc)
        fold_f1m.append(f1_score(y_val, y_p, average='macro',    zero_division=0))
        fold_f1w.append(f1_score(y_val, y_p, average='weighted', zero_division=0))
        fold_acc.append(accuracy_score(y_val, y_p))
        per_f1 = f1_score(y_val, y_p, average=None, zero_division=0)
        for ci, cn in enumerate(CLASS_NAMES):
            per_class[cn].append(per_f1[ci] if ci < len(per_f1) else 0)

    return {
        'Model'              : model_name,
        'F1_Macro_Mean'      : round(np.mean(fold_f1m), 4),
        'F1_Macro_Std'       : round(np.std(fold_f1m),  4),
        'F1_Weighted_Mean'   : round(np.mean(fold_f1w), 4),
        'Accuracy_Mean'      : round(np.mean(fold_acc), 4),
        'F1_Stable_Mean'     : round(np.mean(per_class['Stable']),     4),
        'F1_Challenged_Mean' : round(np.mean(per_class['Challenged']), 4),
        'F1_Critical_Mean'   : round(np.mean(per_class['Critical']),   4),
    }

print('Model A — 15 SKB scale features only:')
res_A = cv_evaluate(X_scale[:, SKB_IDX], y_all, 'Model_A_Scale_Only')
print(f'  F1_Macro={res_A["F1_Macro_Mean"]} ± {res_A["F1_Macro_Std"]}')

print('\nModel B — 15 SKB scale features + 7 demographics:')
X_combined = np.hstack([X_scale[:, SKB_IDX], X_demo])
res_B = cv_evaluate(X_combined, y_all, 'Model_B_Scale_Plus_Demographics')
print(f'  F1_Macro={res_B["F1_Macro_Mean"]} ± {res_B["F1_Macro_Std"]}')

demo_df = pd.DataFrame([res_A, res_B])
os.makedirs(os.path.join('summary', 'Evaluation'), exist_ok=True)
DEMO_PATH = os.path.join('summary', 'Evaluation', 'demographics_experiment.csv')
demo_df.to_csv(DEMO_PATH, index=False)
print(f'\n✓ Saved: {DEMO_PATH}')
print(demo_df.to_string(index=False))

Model A — 15 SKB scale features only:
  F1_Macro=0.8543 ± 0.0254

Model B — 15 SKB scale features + 7 demographics:
  F1_Macro=0.8391 ± 0.0202

✓ Saved: summary\Evaluation\demographics_experiment.csv
                          Model  F1_Macro_Mean  F1_Macro_Std  F1_Weighted_Mean  Accuracy_Mean  F1_Stable_Mean  F1_Challenged_Mean  F1_Critical_Mean
             Model_A_Scale_Only         0.8543        0.0254            0.8985         0.8981          0.7874              0.8377            0.9377
Model_B_Scale_Plus_Demographics         0.8391        0.0202            0.8963         0.8962          0.7432              0.8340            0.9402


## Cell 5 — Ablation Study

Progressively adds pipeline components to quantify each one's contribution to final performance. Four configurations are evaluated via Stratified 5-fold CV:

| Config | Features | SMOTE | XGB Parameters |
|--------|----------|-------|----------------|
| 1 — No pipeline | All 26 scale items | No | Default |
| 2 — Feature selection | SKB 15 features | No | Default |
| 3 — Full baseline | SKB 15 features | Yes | Default |
| 4 — Optimised | SKB 15 features | Yes | Bayesian-optimised (from NB04) |

Config 4 uses the saved Bayesian-optimised XGB model parameters (`models/Optimised/skb/xgb_bayes.pkl`) to show whether hyperparameter tuning adds value beyond the baseline pipeline.

**Note on CV vs test-set F1:** CV F1 scores are computed on naturally imbalanced folds (SMOTE is applied to training splits only, not validation). The held-out test F1 (0.8667) may differ from CV means because the stratified test split's class proportions, combined with SMOTE-rebalanced training, can yield a more favourable distribution for the minority Stable class.

In [5]:
# Load Bayesian-optimised XGB parameters
xgb_bayes    = joblib.load(os.path.join('models', 'Optimised', 'skb', 'xgb_bayes.pkl'))
bayes_params = xgb_bayes.get_params()
print(f'Bayes-optimised XGB params: {bayes_params}')

ablation_results = []

configs = [
    ('Config 1 — No pipeline',
     X_scale, False,
     XGBClassifier(n_estimators=100, random_state=SEED, n_jobs=-1,
                   eval_metric='mlogloss', verbosity=0)),
    ('Config 2 — Feature selection (SKB 15)',
     X_scale[:, SKB_IDX], False,
     XGBClassifier(n_estimators=100, random_state=SEED, n_jobs=-1,
                   eval_metric='mlogloss', verbosity=0)),
    ('Config 3 — Feature selection + SMOTE',
     X_scale[:, SKB_IDX], True,
     XGBClassifier(n_estimators=100, random_state=SEED, n_jobs=-1,
                   eval_metric='mlogloss', verbosity=0)),
    ('Config 4 — Full pipeline + Bayes tuning',
     X_scale[:, SKB_IDX], True,
     XGBClassifier(**{k: v for k, v in bayes_params.items()
                      if k not in ['n_jobs', 'random_state', 'verbosity',
                                   'eval_metric', 'early_stopping_rounds', 'callbacks']},
                   random_state=SEED, n_jobs=-1,
                   eval_metric='mlogloss', verbosity=0)),
]

sm = SMOTE(random_state=SEED)

for config_name, X_cfg, use_smote, xgb_cfg in configs:
    print(f'\n{config_name}')
    cv_local = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    fold_f1m, fold_acc = [], []

    for tr_idx, val_idx in cv_local.split(X_cfg, y_all):
        X_tr, X_val = X_cfg[tr_idx], X_cfg[val_idx]
        y_tr, y_val = y_all[tr_idx], y_all[val_idx]

        if use_smote:
            X_tr, y_tr = sm.fit_resample(X_tr, y_tr)

        sc = StandardScaler()
        X_tr_sc  = sc.fit_transform(X_tr)
        X_val_sc = sc.transform(X_val)

        clf = copy.deepcopy(xgb_cfg)
        clf.fit(X_tr_sc, y_tr)
        y_p = clf.predict(X_val_sc)
        fold_f1m.append(f1_score(y_val, y_p, average='macro',    zero_division=0))
        fold_acc.append(accuracy_score(y_val, y_p))

    row = {
        'Configuration'  : config_name,
        'F1_Macro_Mean'  : round(np.mean(fold_f1m), 4),
        'F1_Macro_Std'   : round(np.std(fold_f1m),  4),
        'Accuracy_Mean'  : round(np.mean(fold_acc), 4),
        'SMOTE'          : use_smote,
        'N_Features'     : X_cfg.shape[1],
    }
    ablation_results.append(row)
    print(f'  F1_Macro = {row["F1_Macro_Mean"]} ± {row["F1_Macro_Std"]}')

ablation_df = pd.DataFrame(ablation_results)
os.makedirs(os.path.join('summary', 'Evaluation'), exist_ok=True)
ABL_PATH    = os.path.join('summary', 'Evaluation', 'ablation_study.csv')
ablation_df.to_csv(ABL_PATH, index=False)
print(f'\n✓ Saved: {ABL_PATH}')
print(ablation_df[['Configuration','F1_Macro_Mean','F1_Macro_Std',
                    'Accuracy_Mean','SMOTE','N_Features']].to_string(index=False))

Bayes-optimised XGB params: {'objective': 'multi:softprob', 'base_score': None, 'booster': None, 'callbacks': None, 'colsample_bylevel': None, 'colsample_bynode': None, 'colsample_bytree': 1.0, 'device': None, 'early_stopping_rounds': None, 'enable_categorical': True, 'eval_metric': 'mlogloss', 'feature_types': None, 'feature_weights': None, 'gamma': None, 'grow_policy': None, 'importance_type': None, 'interaction_constraints': None, 'learning_rate': 0.03759787023879407, 'max_bin': None, 'max_cat_threshold': None, 'max_cat_to_onehot': None, 'max_delta_step': None, 'max_depth': 5, 'max_leaves': None, 'min_child_weight': None, 'missing': nan, 'monotone_constraints': None, 'multi_strategy': None, 'n_estimators': 300, 'n_jobs': 1, 'num_parallel_tree': None, 'random_state': 42, 'reg_alpha': None, 'reg_lambda': None, 'sampling_method': None, 'scale_pos_weight': None, 'subsample': 0.6, 'tree_method': None, 'validate_parameters': None, 'verbosity': 0}

Config 1 — No pipeline
  F1_Macro = 0.869

## Cell 6 — Multi-Output Classification Experiment

Instead of predicting a single `Mental Health Status` label, this experiment trains a `MultiOutputClassifier(XGB)` to simultaneously predict all three per-scale severity levels: `Stress Level`, `Anxiety Level`, and `Depression Level` (each encoded as Stable=0, Challenged=1, Critical=2).

`MultiOutputClassifier` fits one independent XGB per output — it does not model output correlations. SMOTE is not applied because it is designed for single-output targets; applying it here would require a multi-label oversampling strategy.

**Metrics:**
- **Per-output F1_Macro:** macro F1 for each of the three scale predictions independently
- **Exact Match Ratio (Subset Accuracy):** fraction of samples where all three outputs are correctly predicted simultaneously
- **Hamming Loss:** fraction of output-label pairs that are incorrectly predicted (lower is better)

Results are compared against the single-output baseline (XGB + SKB, same 5-fold CV, no SMOTE for a fair comparison).

In [6]:
# Multi-output targets
LEVEL_MAP  = {'Stable': 0, 'Challenged': 1, 'Critical': 2}
y_stress   = df['Stress Level'].map(LEVEL_MAP).values
y_anxiety  = df['Anxiety Level'].map(LEVEL_MAP).values
y_depress  = df['Depression Level'].map(LEVEL_MAP).values
Y_multi    = np.column_stack([y_stress, y_anxiety, y_depress])  # (2022, 3)

X_skb_all = X_scale[:, SKB_IDX]   # 15 SKB features for all 2022 samples

cv_local = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

# Single-output CV (baseline comparison — no SMOTE for fair comparison)
fold_f1m_single, fold_acc_single = [], []

# Multi-output CV
fold_f1_stress, fold_f1_anxiety, fold_f1_depress = [], [], []
fold_hamming, fold_exact = [], []

for tr_idx, val_idx in cv_local.split(X_skb_all, y_all):
    X_tr, X_val = X_skb_all[tr_idx], X_skb_all[val_idx]
    sc = StandardScaler()
    X_tr_sc  = sc.fit_transform(X_tr)
    X_val_sc = sc.transform(X_val)

    # Single-output
    y_tr_s, y_val_s = y_all[tr_idx], y_all[val_idx]
    clf_s = XGBClassifier(n_estimators=100, random_state=SEED, n_jobs=-1,
                          eval_metric='mlogloss', verbosity=0)
    clf_s.fit(X_tr_sc, y_tr_s)
    y_p_s = clf_s.predict(X_val_sc)
    fold_f1m_single.append(f1_score(y_val_s, y_p_s, average='macro', zero_division=0))
    fold_acc_single.append(accuracy_score(y_val_s, y_p_s))

    # Multi-output
    Y_tr, Y_val = Y_multi[tr_idx], Y_multi[val_idx]
    clf_m = MultiOutputClassifier(
        XGBClassifier(n_estimators=100, random_state=SEED, n_jobs=-1,
                      eval_metric='mlogloss', verbosity=0)
    )
    clf_m.fit(X_tr_sc, Y_tr)
    Y_p = clf_m.predict(X_val_sc)

    fold_f1_stress.append(f1_score(Y_val[:,0], Y_p[:,0], average='macro', zero_division=0))
    fold_f1_anxiety.append(f1_score(Y_val[:,1], Y_p[:,1], average='macro', zero_division=0))
    fold_f1_depress.append(f1_score(Y_val[:,2], Y_p[:,2], average='macro', zero_division=0))
    fold_hamming.append(float(np.mean(Y_val != Y_p)))
    fold_exact.append(np.mean(np.all(Y_val == Y_p, axis=1)))

multi_results = [
    {'Approach': 'Single-Output (XGB+SKB, no SMOTE)',
     'F1_Macro_MHS'       : round(np.mean(fold_f1m_single), 4),
     'Accuracy_MHS'       : round(np.mean(fold_acc_single), 4),
     'F1_Macro_Stress'    : 'N/A',
     'F1_Macro_Anxiety'   : 'N/A',
     'F1_Macro_Depression': 'N/A',
     'Hamming_Loss'       : 'N/A',
     'Exact_Match_Ratio'  : 'N/A'},
    {'Approach': 'Multi-Output (XGB, no SMOTE)',
     'F1_Macro_MHS'       : 'N/A',
     'Accuracy_MHS'       : 'N/A',
     'F1_Macro_Stress'    : round(np.mean(fold_f1_stress),  4),
     'F1_Macro_Anxiety'   : round(np.mean(fold_f1_anxiety), 4),
     'F1_Macro_Depression': round(np.mean(fold_f1_depress), 4),
     'Hamming_Loss'       : round(np.mean(fold_hamming), 4),
     'Exact_Match_Ratio'  : round(np.mean(fold_exact),  4)},
]

multi_df  = pd.DataFrame(multi_results)
os.makedirs(os.path.join('summary', 'Evaluation'), exist_ok=True)
MULTI_PATH = os.path.join('summary', 'Evaluation', 'multi_output_experiment.csv')
multi_df.to_csv(MULTI_PATH, index=False)
print(f'✓ Saved: {MULTI_PATH}')
print(multi_df.to_string(index=False))
print()
print('── Notebook 07 complete ──')
print('  figures/Evaluation/          roc_auc_curves.png + calibration_curves.png')
print('  summary/Evaluation/          statistical_tests.csv')
print('                               demographics_experiment.csv')
print('                               ablation_study.csv')
print('                               multi_output_experiment.csv')

✓ Saved: summary\Evaluation\multi_output_experiment.csv
                         Approach F1_Macro_MHS Accuracy_MHS F1_Macro_Stress F1_Macro_Anxiety F1_Macro_Depression Hamming_Loss Exact_Match_Ratio
Single-Output (XGB+SKB, no SMOTE)       0.8528       0.8996             N/A              N/A                 N/A          N/A               N/A
     Multi-Output (XGB, no SMOTE)          N/A          N/A          0.6589           0.8922              0.8719       0.1123            0.7038

── Notebook 07 complete ──
  figures/Evaluation/          roc_auc_curves.png + calibration_curves.png
  summary/Evaluation/          statistical_tests.csv
                               demographics_experiment.csv
                               ablation_study.csv
                               multi_output_experiment.csv
